# 1.Package imports section

In [0]:
import re
import logging
import pyspark.sql.functions as F


# 2.Dataset configurations

In [0]:
# logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_CPI")

#silver tables config
ds_config = {
        "silver_table": "cpt_utility_catalog.silver.silver_cpi_cleaned",
        "bronze_table": "cpt_utility_catalog.bronze.bronze_cpi_raw",
        "changes": {
            "column_mapping": {
                "H01": "survey_code",
                "H03": "series_identifier",
                "H04": "category",
                "H05": "subcategory",
                "H06": "decile",
                "H13": "geographic_area",
                "H17": "unit_of_measure",
                "H18": "base_period",
                "H24": "start_period",
                "H25": "frequency",
            },
            "data_types":{
                "survey_code": "string",
                "series_identifier": "string",
                "category": "string",
                "subcategory": "string",
                "decile": "int",
                "geographic_area": "string",
                "unit_of_measure": "string",
                "base_period": "string",
                "start_period": "date",
                "frequency": "string",
            },
            "date_header_mapping":{
                "pattern": "^MO\\d{6}",
                "after_pattern": "\\d{4}-\\d{2}",
                "format": "yyyy-MM",
                "data_type": "decimal(5,1)",
            },
            "trim_whitespace": True,
            "drop_columns": ["H02"],
            "fill_na_value": None,  
            "drop_duplicates": True,
        },
    }

df_new = spark.read.table(ds_config["bronze_table"])
changes = ds_config["changes"]

logger.info("Silver layer CPI table configuration loaded")

# 3. Dataset Cleaning

## 3.1 Drop Columns

In [0]:
logger.info("3.1 Droping column(s)")
# obtain list of columns to delete
drop_columns = changes.get("drop_columns", [])
# iterate through list of columns to delete and delete
if isinstance(drop_columns, list):
    df_new = df_new.drop(*drop_columns)

logger.info(f"\t- Column(s) successfully dropped")

##3.2 Clean Column Names

In [0]:
logger.info("3.2 Renaming column(s)")
# obtain dictionary of columns to rename and rename them 
column_mapping = changes.get("column_mapping", {})
if isinstance(column_mapping, dict):
    for old_col, new_col in column_mapping.items():
        df_new = df_new.withColumnRenamed(old_col, new_col)
# obtain dated header dictionary and rename to yyyy-MM format
date_header_mapping = changes.get("date_header_mapping", {})
if isinstance(date_header_mapping, dict):
    pattern = date_header_mapping.get("pattern", "")
    
    for col in df_new.columns:
        if re.match(pattern, col):
            new_name = f"{col[4:]}-{col[2:4]}"
            df_new = df_new.withColumnRenamed(col, new_name)

logger.info("\t- Column(s) renamed")

## 3.3 Trim Whitespace

In [0]:
logger.info("3.3 Trimming whitespace(s)")
# trim whitespace on all row cells sicne it's all string, before we cast type
trim = changes["trim_whitespace"]
if trim:
    df_new = df_new.select([F.trim(F.col(col)).alias(col) if data_type == "string" else F.col(col) for col, data_type in df_new.dtypes])
logger.info("\t- Whitespace(s) trimmed")

## 3.4 Clean Malformed Column Rows

In [0]:
logger.info("3.4 Cleaning Malformed Column row(s)")
# turning start_period into convertable format of date data type
df_new = df_new.withColumn("start_period", F.translate(F.col("start_period"), " ", "-"))
logger.info("\t- Malformed column row(s) cleaned")

## 3.5 Cast Data Types

In [0]:
logger.info("3.5 Casting data type(s)")
dt_transformations = {}
dt_types_map = changes.get("data_types",{})
date_header_map = changes.get("date_header_mapping", {})

# cast data types of fixed column names
if isinstance(dt_types_map, dict):
    for col, data_type in dt_types_map.items():
        dt_transformations[col] = F.col(col).cast(data_type)

# cast data types of date columns which increase over time
if isinstance(date_header_map, dict):
    date_header_data_type = date_header_map["data_type"]
    date_header_pattern = date_header_map["after_pattern"]

    for col in df_new.columns:
        if re.match(date_header_pattern, col):
            dt_transformations[col] = F.col(col).cast(date_header_data_type)

df_new = df_new.withColumns(dt_transformations)
logger.info("\t- Data type(s) casted")

## 3.6 Drop Duplicates

In [0]:
logger.info("3.6 Dropping duplicates")
df_new = df_new.dropDuplicates()
logger.info("\t- Duplicate(s) dropped")

# Writing To Silver Layer

In [0]:
logger.info("3.7 Writing to Silver Table")
(df_new.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(ds_config["silver_table"]))
logger.info("\t- Data written to Silver Table")